In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/sales_master.csv")
df["date"] = pd.to_datetime(df["date"])

weekly = (
    df.groupby(["sku_id", pd.Grouper(key="date", freq="W-MON")])["units_sold"]
    .sum()
    .reset_index()
)
weekly = weekly.rename(columns={"date": "week_start"})
print(weekly.shape)
weekly.head(10)

(18059, 3)


,sku_id,week_start,units_sold
0,SKU0001,2026-06-08,1.0
1,SKU0001,2026-06-15,19.0
2,SKU0001,2026-06-22,26.0
3,SKU0001,2026-06-29,24.0
4,SKU0001,2026-07-06,18.0
5,SKU0001,2026-07-13,15.0
6,SKU0001,2026-07-20,14.0
7,SKU0002,2024-08-05,17.0
8,SKU0002,2024-08-12,23.0
9,SKU0002,2024-08-19,27.0


In [2]:
weekly = weekly.sort_values(["sku_id", "week_start"])

weekly["baseline_forecast"] = weekly.groupby("sku_id")["units_sold"].shift(52)

weekly.head(15)

,sku_id,week_start,units_sold,baseline_forecast
0,SKU0001,2026-06-08,1.0,NaN
1,SKU0001,2026-06-15,19.0,NaN
2,SKU0001,2026-06-22,26.0,NaN
3,SKU0001,2026-06-29,24.0,NaN
4,SKU0001,2026-07-06,18.0,NaN
5,SKU0001,2026-07-13,15.0,NaN
6,SKU0001,2026-07-20,14.0,NaN
7,SKU0002,2024-08-05,17.0,NaN
8,SKU0002,2024-08-12,23.0,NaN
9,SKU0002,2024-08-19,27.0,NaN


In [3]:
sku0002 = weekly[weekly["sku_id"] == "SKU0002"]
sku0002.iloc[50:58]

,sku_id,week_start,units_sold,baseline_forecast
57,SKU0002,2025-07-21,17.0,NaN
58,SKU0002,2025-07-28,22.0,NaN
59,SKU0002,2025-08-04,24.0,17.0
60,SKU0002,2025-08-11,21.0,23.0
61,SKU0002,2025-08-18,19.0,27.0
62,SKU0002,2025-08-25,20.0,23.0
63,SKU0002,2025-09-01,22.0,30.0
64,SKU0002,2025-09-08,25.0,23.0


In [4]:
valid = weekly.dropna(subset=["baseline_forecast"])

wape = (valid["units_sold"] - valid["baseline_forecast"]).abs().sum() / valid["units_sold"].sum()
print(f"Baseline WAPE: {wape:.3f} ({wape*100:.1f}%)")

Baseline WAPE: 0.142 (14.2%)


In [5]:
weekly["lag_1"] = weekly.groupby("sku_id")["units_sold"].shift(1)
weekly["lag_2"] = weekly.groupby("sku_id")["units_sold"].shift(2)
weekly["lag_4"] = weekly.groupby("sku_id")["units_sold"].shift(4)

weekly[weekly["sku_id"] == "SKU0002"].iloc[50:58]

,sku_id,week_start,units_sold,baseline_forecast,lag_1,lag_2,lag_4
57,SKU0002,2025-07-21,17.0,NaN,19.0,16.0,38.0
58,SKU0002,2025-07-28,22.0,NaN,17.0,19.0,42.0
59,SKU0002,2025-08-04,24.0,17.0,22.0,17.0,16.0
60,SKU0002,2025-08-11,21.0,23.0,24.0,22.0,19.0
61,SKU0002,2025-08-18,19.0,27.0,21.0,24.0,17.0
62,SKU0002,2025-08-25,20.0,23.0,19.0,21.0,22.0
63,SKU0002,2025-09-01,22.0,30.0,20.0,19.0,24.0
64,SKU0002,2025-09-08,25.0,23.0,22.0,20.0,21.0


In [6]:
sales_master = pd.read_csv("../data/processed/sales_master.csv")
sales_master["date"] = pd.to_datetime(sales_master["date"])

weekly_calendar = (
    sales_master.groupby(["sku_id", pd.Grouper(key="date", freq="W-MON")])
    .agg(month=("month", "first"), season=("season", "first"), promo_flag=("promo_flag", "max"))
    .reset_index()
    .rename(columns={"date": "week_start"})
)

weekly = weekly.merge(weekly_calendar, on=["sku_id", "week_start"], how="left")
weekly.head()

,sku_id,week_start,units_sold,baseline_forecast,lag_1,lag_2,lag_4,month,season,promo_flag
0,SKU0001,2026-06-08,1.0,NaN,NaN,NaN,NaN,6,Monsoon,0
1,SKU0001,2026-06-15,19.0,NaN,1.0,NaN,NaN,6,Monsoon,1
2,SKU0001,2026-06-22,26.0,NaN,19.0,1.0,NaN,6,Monsoon,1
3,SKU0001,2026-06-29,24.0,NaN,26.0,19.0,NaN,6,Monsoon,1
4,SKU0001,2026-07-06,18.0,NaN,24.0,26.0,1.0,7,Monsoon,1


In [7]:
weekly["rolling_mean_4"] = (
    weekly.groupby("sku_id")["units_sold"]
    .transform(lambda s: s.shift(1).rolling(window=4).mean())
)

weekly[weekly["sku_id"] == "SKU0002"].iloc[55:62]

,sku_id,week_start,units_sold,baseline_forecast,lag_1,lag_2,lag_4,month,season,promo_flag,rolling_mean_4
62,SKU0002,2025-08-25,20.0,23.0,19.0,21.0,22.0,8,Monsoon,0,21.50
63,SKU0002,2025-09-01,22.0,30.0,20.0,19.0,24.0,8,Monsoon,0,21.00
64,SKU0002,2025-09-08,25.0,23.0,22.0,20.0,21.0,9,Monsoon,0,20.50
65,SKU0002,2025-09-15,24.0,23.0,25.0,22.0,19.0,9,Monsoon,0,21.50
66,SKU0002,2025-09-22,21.0,20.0,24.0,25.0,20.0,9,Monsoon,0,22.75
67,SKU0002,2025-09-29,32.0,33.0,21.0,24.0,22.0,9,Monsoon,0,23.00
68,SKU0002,2025-10-06,38.0,34.0,32.0,21.0,25.0,10,Autumn,0,25.50


In [8]:
print(weekly.columns.tolist())
print(weekly.shape)

['sku_id', 'week_start', 'units_sold', 'baseline_forecast', 'lag_1', 'lag_2', 'lag_4', 'month', 'season', 'promo_flag', 'rolling_mean_4']
(18059, 11)


In [9]:
model_data = weekly.dropna(subset=["lag_1", "lag_2", "lag_4", "rolling_mean_4"]).copy()
print("Rows before:", weekly.shape[0])
print("Rows after dropping incomplete features:", model_data.shape[0])

Rows before: 18059
Rows after dropping incomplete features: 17260


In [10]:
cutoff_date = model_data["week_start"].max() - pd.Timedelta(weeks=8)

train = model_data[model_data["week_start"] <= cutoff_date]
test = model_data[model_data["week_start"] > cutoff_date]

print("Train rows:", train.shape[0], "| up to:", train["week_start"].max())
print("Test rows:", test.shape[0], "| from:", test["week_start"].min())

Train rows: 15709 | up to: 2026-05-25 00:00:00
Test rows: 1551 | from: 2026-06-01 00:00:00


In [11]:
import lightgbm as lgb

feature_cols = ["lag_1", "lag_2", "lag_4", "rolling_mean_4", "month", "promo_flag"]

X_train = train[feature_cols]
y_train = train["units_sold"]
X_test = test[feature_cols]
y_test = test["units_sold"]

model = lgb.LGBMRegressor(random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully.")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000800 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1035
[LightGBM] [Info] Number of data points in the train set: 15709, number of used features: 6
[LightGBM] [Info] Start training from score 58.155962
Model trained successfully.


In [12]:
predictions = model.predict(X_test)

wape_model = np.abs(y_test - predictions).sum() / y_test.sum()
print(f"Model WAPE: {wape_model:.3f} ({wape_model*100:.1f}%)")
print(f"Baseline WAPE: 0.142 (14.2%)")

if wape_model < 0.142:
    print(f"\nModel BEATS the baseline by {(0.142 - wape_model)*100:.1f} percentage points.")
else:
    print(f"\nModel does NOT beat the baseline. Difference: {(wape_model - 0.142)*100:.1f} percentage points.")

Model WAPE: 0.197 (19.7%)
Baseline WAPE: 0.142 (14.2%)

Model does NOT beat the baseline. Difference: 5.5 percentage points.


In [13]:
model_data = weekly.dropna(subset=["lag_1", "lag_2", "lag_4", "rolling_mean_4"]).copy()
model_data["sku_id"] = model_data["sku_id"].astype("category")

cutoff_date = model_data["week_start"].max() - pd.Timedelta(weeks=8)
train = model_data[model_data["week_start"] <= cutoff_date]
test = model_data[model_data["week_start"] > cutoff_date]

feature_cols = ["sku_id", "lag_1", "lag_2", "lag_4", "rolling_mean_4", "month", "promo_flag"]

X_train = train[feature_cols]
y_train = train["units_sold"]
X_test = test[feature_cols]
y_test = test["units_sold"]

model2 = lgb.LGBMRegressor(random_state=42)
model2.fit(X_train, y_train)

predictions2 = model2.predict(X_test)
wape_model2 = np.abs(y_test - predictions2).sum() / y_test.sum()
print(f"Model WAPE (with sku_id): {wape_model2:.3f} ({wape_model2*100:.1f}%)")
print(f"Baseline WAPE: 0.142 (14.2%)")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000508 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1219
[LightGBM] [Info] Number of data points in the train set: 15709, number of used features: 7
[LightGBM] [Info] Start training from score 58.155962
Model WAPE (with sku_id): 0.187 (18.7%)
Baseline WAPE: 0.142 (14.2%)


In [14]:
model_data = weekly.dropna(subset=["lag_1", "lag_2", "lag_4", "rolling_mean_4", "baseline_forecast"]).copy()
model_data["sku_id"] = model_data["sku_id"].astype("category")

cutoff_date = model_data["week_start"].max() - pd.Timedelta(weeks=8)
train = model_data[model_data["week_start"] <= cutoff_date]
test = model_data[model_data["week_start"] > cutoff_date]

feature_cols = ["sku_id", "baseline_forecast", "lag_1", "lag_2", "lag_4", "rolling_mean_4", "month", "promo_flag"]

X_train = train[feature_cols]
y_train = train["units_sold"]
X_test = test[feature_cols]
y_test = test["units_sold"]

model3 = lgb.LGBMRegressor(random_state=42)
model3.fit(X_train, y_train)

predictions3 = model3.predict(X_test)
wape_model3 = np.abs(y_test - predictions3).sum() / y_test.sum()
print(f"Model WAPE (with baseline as feature): {wape_model3:.3f} ({wape_model3*100:.1f}%)")
print(f"Baseline WAPE: 0.142 (14.2%)")
print(f"Train rows: {len(train)}, Test rows: {len(test)}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000548 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1459
[LightGBM] [Info] Number of data points in the train set: 7434, number of used features: 8
[LightGBM] [Info] Start training from score 58.890369
Model WAPE (with baseline as feature): 0.147 (14.7%)
Baseline WAPE: 0.142 (14.2%)
Train rows: 7434, Test rows: 1360


In [15]:
def compute_wape(actual, predicted):
    return np.abs(actual - predicted).sum() / actual.sum()

model_data = weekly.dropna(subset=["lag_1", "lag_2", "lag_4", "rolling_mean_4", "baseline_forecast"]).copy()
model_data["sku_id"] = model_data["sku_id"].astype("category")

feature_cols = ["sku_id", "baseline_forecast", "lag_1", "lag_2", "lag_4", "rolling_mean_4", "month", "promo_flag"]

all_weeks = sorted(model_data["week_start"].unique())
n_folds = 3
horizon = 8

results = []

for fold in range(n_folds):
    test_end_idx = len(all_weeks) - fold * horizon
    test_start_idx = test_end_idx - horizon
    if test_start_idx <= 0:
        break

    test_weeks = all_weeks[test_start_idx:test_end_idx]
    train_weeks = all_weeks[:test_start_idx]

    train_fold = model_data[model_data["week_start"].isin(train_weeks)]
    test_fold = model_data[model_data["week_start"].isin(test_weeks)]

    m = lgb.LGBMRegressor(random_state=42, verbose=-1)
    m.fit(train_fold[feature_cols], train_fold["units_sold"])
    preds = m.predict(test_fold[feature_cols])

    results.append({
        "fold": fold,
        "test_start": test_weeks[0],
        "test_end": test_weeks[-1],
        "model_wape": compute_wape(test_fold["units_sold"], preds),
        "baseline_wape": compute_wape(test_fold["units_sold"], test_fold["baseline_forecast"]),
        "model_bias": (test_fold["units_sold"] - preds).mean(),
    })

results_df = pd.DataFrame(results)
print(results_df)
print(f"\nAverage model WAPE: {results_df['model_wape'].mean():.3f}")
print(f"Average baseline WAPE: {results_df['baseline_wape'].mean():.3f}")
print(f"Average model bias: {results_df['model_bias'].mean():.3f} (positive = under-forecasting)")

   fold test_start   test_end  model_wape  baseline_wape  model_bias
0     0 2026-06-01 2026-07-20    0.146770       0.134661   -3.129495
1     1 2026-04-06 2026-05-25    0.129170       0.145870   -1.161186
2     2 2026-02-09 2026-03-30    0.140526       0.152986    1.904650

Average model WAPE: 0.139
Average baseline WAPE: 0.145
Average model bias: -0.795 (positive = under-forecasting)


In [16]:
final_model = lgb.LGBMRegressor(random_state=42, verbose=-1)
final_model.fit(model_data[feature_cols], model_data["units_sold"])

model_data["predicted_demand"] = final_model.predict(model_data[feature_cols])

latest_forecast = (
    model_data.sort_values("week_start")
    .groupby("sku_id")
    .tail(1)[["sku_id", "week_start", "predicted_demand"]]
)

latest_forecast.to_csv("../data/processed/latest_forecast.csv", index=False)
print(latest_forecast.shape)
latest_forecast.head()

(170, 3)


,sku_id,week_start,predicted_demand
17849,SKU0198,2026-07-20,154.349267
17744,SKU0197,2026-07-20,18.327841
17954,SKU0199,2026-07-20,11.610890
10230,SKU0115,2026-07-20,35.855945
17535,SKU0195,2026-07-20,202.001534


In [17]:
all_skus = weekly["sku_id"].unique()
forecasted_skus = latest_forecast["sku_id"].unique()
missing_skus = set(all_skus) - set(forecasted_skus)

print(f"Total SKUs: {len(all_skus)}")
print(f"SKUs with a forecast: {len(forecasted_skus)}")
print(f"SKUs missing a forecast: {len(missing_skus)}")

sku_clean = pd.read_csv("../data/processed/sku_master_clean.csv")
sku_clean[sku_clean["sku_id"].isin(missing_skus)][["sku_id", "launch_date"]].head(10)

Total SKUs: 200
SKUs with a forecast: 170
SKUs missing a forecast: 30


,sku_id,launch_date
0,SKU0001,2026-06-08
10,SKU0011,2026-03-31
16,SKU0017,2026-03-27
21,SKU0022,2026-05-13
22,SKU0023,2026-03-08
24,SKU0025,2026-03-04
33,SKU0034,2026-04-18
35,SKU0036,2026-04-28
43,SKU0044,2026-02-09
57,SKU0058,2026-05-25


In [18]:
sku_clean = pd.read_csv("../data/processed/sku_master_clean.csv")

# recent average demand per SKU (last 4 available weeks), for SKUs with SOME history but not enough for baseline_forecast
recent_avg = (
    weekly.sort_values("week_start")
    .groupby("sku_id")
    .tail(4)
    .groupby("sku_id")["units_sold"]
    .mean()
    .reset_index()
    .rename(columns={"units_sold": "recent_avg_demand"})
)

# category average, as a last-resort fallback for brand new SKUs with barely any history
recent_avg = recent_avg.merge(sku_clean[["sku_id", "category"]], on="sku_id")
category_avg = recent_avg.groupby("category")["recent_avg_demand"].mean().reset_index()
category_avg = category_avg.rename(columns={"recent_avg_demand": "category_avg_demand"})

missing_forecast = recent_avg[recent_avg["sku_id"].isin(missing_skus)].merge(category_avg, on="category")
missing_forecast["predicted_demand"] = missing_forecast["recent_avg_demand"]
missing_forecast["week_start"] = weekly["week_start"].max()
missing_forecast["is_low_confidence"] = True

missing_forecast = missing_forecast[["sku_id", "week_start", "predicted_demand", "is_low_confidence"]]
print(missing_forecast.shape)
missing_forecast.head()

(30, 4)


,sku_id,week_start,predicted_demand,is_low_confidence
0,SKU0001,2026-07-20,17.75,True
1,SKU0011,2026-07-20,2.25,True
2,SKU0017,2026-07-20,33.50,True
3,SKU0022,2026-07-20,66.50,True
4,SKU0023,2026-07-20,73.50,True


In [19]:
latest_forecast["is_low_confidence"] = False

full_forecast = pd.concat([latest_forecast, missing_forecast], ignore_index=True)

print("Total SKUs covered:", full_forecast["sku_id"].nunique())
print("Low confidence count:", full_forecast["is_low_confidence"].sum())

full_forecast.to_csv("../data/processed/latest_forecast.csv", index=False)
full_forecast.head()

Total SKUs covered: 200
Low confidence count: 30


,sku_id,week_start,predicted_demand,is_low_confidence
0,SKU0198,2026-07-20,154.349267,False
1,SKU0197,2026-07-20,18.327841,False
2,SKU0199,2026-07-20,11.610890,False
3,SKU0115,2026-07-20,35.855945,False
4,SKU0195,2026-07-20,202.001534,False


In [20]:
# Step 1: lookup table of all real historical demand, and the two SKU groups
history_lookup = weekly.set_index(["sku_id", "week_start"])["units_sold"].to_dict()

full_history_skus = latest_forecast["sku_id"].unique()
low_confidence_skus = missing_forecast["sku_id"].unique()

print("Full-history SKUs (will get a real recursive forecast):", len(full_history_skus))
print("Low-confidence SKUs (will get the flat fallback):", len(low_confidence_skus))
print("Total weekly history rows available for lookup:", len(history_lookup))

Full-history SKUs (will get a real recursive forecast): 170
Low-confidence SKUs (will get the flat fallback): 30
Total weekly history rows available for lookup: 18059


In [21]:
def forecast_sku_horizon(sku_id, horizon=8):
    sku_hist = weekly[weekly["sku_id"] == sku_id].sort_values("week_start")
    last_week = sku_hist["week_start"].max()

    # last 4 actual weekly demand values, oldest to newest
    recent_demand = list(sku_hist["units_sold"].tail(4))

    results = []
    for h in range(1, horizon + 1):
        target_week = last_week + pd.Timedelta(weeks=h)

        lag_1 = recent_demand[-1]
        lag_2 = recent_demand[-2]
        lag_4 = recent_demand[-4]
        rolling_mean_4 = sum(recent_demand[-4:]) / 4

        month = target_week.month
        promo_flag = 0  # no future promo calendar available — documented assumption

        baseline_forecast = history_lookup.get((sku_id, target_week - pd.Timedelta(weeks=52)))
        if baseline_forecast is None:
            baseline_forecast = rolling_mean_4  # fallback if no year-ago data exists

        row = pd.DataFrame([{
            "sku_id": sku_id,
            "baseline_forecast": baseline_forecast,
            "lag_1": lag_1,
            "lag_2": lag_2,
            "lag_4": lag_4,
            "rolling_mean_4": rolling_mean_4,
            "month": month,
            "promo_flag": promo_flag,
        }])
        row["sku_id"] = row["sku_id"].astype(model_data["sku_id"].dtype)

        pred = max(final_model.predict(row[feature_cols])[0], 0)  # demand can't be negative

        results.append({
            "sku_id": sku_id,
            "week_start": target_week,
            "horizon_week": h,
            "predicted_demand": pred
        })

        recent_demand.append(pred)

    return pd.DataFrame(results)

# Test on just one SKU first, before running it 170 times
test_result = forecast_sku_horizon(full_history_skus[0])
print(test_result)

    sku_id week_start  horizon_week  predicted_demand
0  SKU0198 2026-07-27             1        159.603282
1  SKU0198 2026-08-03             2        153.810393
2  SKU0198 2026-08-10             3        162.039212
3  SKU0198 2026-08-17             4        153.430247
4  SKU0198 2026-08-24             5        157.544353
5  SKU0198 2026-08-31             6        158.890348
6  SKU0198 2026-09-07             7        162.545652
7  SKU0198 2026-09-14             8        163.518927


In [22]:
horizon_results = [forecast_sku_horizon(sku) for sku in full_history_skus]
horizon_forecast = pd.concat(horizon_results, ignore_index=True)

print(horizon_forecast.shape)
horizon_forecast.head(16)

(1360, 4)


,sku_id,week_start,horizon_week,predicted_demand
0,SKU0198,2026-07-27,1,159.603282
1,SKU0198,2026-08-03,2,153.810393
2,SKU0198,2026-08-10,3,162.039212
3,SKU0198,2026-08-17,4,153.430247
4,SKU0198,2026-08-24,5,157.544353
5,SKU0198,2026-08-31,6,158.890348
6,SKU0198,2026-09-07,7,162.545652
7,SKU0198,2026-09-14,8,163.518927
8,SKU0197,2026-07-27,1,14.161037
9,SKU0197,2026-08-03,2,17.158535


In [23]:
last_week = weekly["week_start"].max()

low_confidence_rows = []
for _, row in missing_forecast.iterrows():
    for h in range(1, 9):
        low_confidence_rows.append({
            "sku_id": row["sku_id"],
            "week_start": last_week + pd.Timedelta(weeks=h),
            "horizon_week": h,
            "predicted_demand": row["predicted_demand"]
        })

low_confidence_horizon = pd.DataFrame(low_confidence_rows)
print(low_confidence_horizon.shape)
low_confidence_horizon.head(10)

(240, 4)


,sku_id,week_start,horizon_week,predicted_demand
0,SKU0001,2026-07-27,1,17.75
1,SKU0001,2026-08-03,2,17.75
2,SKU0001,2026-08-10,3,17.75
3,SKU0001,2026-08-17,4,17.75
4,SKU0001,2026-08-24,5,17.75
5,SKU0001,2026-08-31,6,17.75
6,SKU0001,2026-09-07,7,17.75
7,SKU0001,2026-09-14,8,17.75
8,SKU0011,2026-07-27,1,2.25
9,SKU0011,2026-08-03,2,2.25


In [24]:
horizon_forecast["is_low_confidence"] = False
low_confidence_horizon["is_low_confidence"] = True

full_horizon_forecast = pd.concat([horizon_forecast, low_confidence_horizon], ignore_index=True)

print(full_horizon_forecast.shape)
print("Unique SKUs:", full_horizon_forecast["sku_id"].nunique())
print("Low confidence rows:", full_horizon_forecast["is_low_confidence"].sum())

full_horizon_forecast.to_csv("../data/processed/horizon_forecast.csv", index=False)
full_horizon_forecast.head()

(1600, 5)
Unique SKUs: 200
Low confidence rows: 240


,sku_id,week_start,horizon_week,predicted_demand,is_low_confidence
0,SKU0198,2026-07-27,1,159.603282,False
1,SKU0198,2026-08-03,2,153.810393,False
2,SKU0198,2026-08-10,3,162.039212,False
3,SKU0198,2026-08-17,4,153.430247,False
4,SKU0198,2026-08-24,5,157.544353,False


In [25]:
weekly[["sku_id", "week_start", "units_sold"]].to_csv("../data/processed/weekly_demand.csv", index=False)
print("Saved:", weekly[["sku_id", "week_start", "units_sold"]].shape)

Saved: (18059, 3)
